In [13]:
# HİBRİT ÖNERİ SİSTEMİ 
import pandas as pd
import numpy as np
import gc

movie = pd.read_csv('/content/ml-latest-small/movies.csv')
rating = pd.read_csv('/content/ml-latest-small/ratings.csv')
df = rating.merge(movie, on='movieId', how='left')

# 500 film + 1000 kullanıcı 
top_500 = df["title"].value_counts().head(500).index
df = df[df["title"].isin(top_500)]
top_users_list = df["userId"].value_counts().head(1000).index
df = df[df["userId"].isin(top_users_list)]
gc.collect()

user_movie_df = df.pivot_table(index="userId", columns="title", values="rating")
random_user = user_movie_df.notna().sum(axis=1).idxmax()
print(f"Seçilen kullanıcı: {random_user}")

# --- USER BASED ---
random_user_df = user_movie_df[user_movie_df.index == random_user]
movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()
movies_watched_df = user_movie_df[movies_watched]
user_movie_count = movies_watched_df.T.notnull().sum().reset_index()
user_movie_count.columns = ["userId", "movie_count"]
perc = len(movies_watched) * 50 / 100
users_same = user_movie_count[user_movie_count["movie_count"] > perc]["userId"]
final_df = movies_watched_df[movies_watched_df.index.isin(users_same)]
corr_df = pd.DataFrame(final_df.T.corrwith(final_df.loc[random_user]), columns=["corr"]).reset_index()
corr_df.columns = ["userId", "corr"]
top_users = corr_df[(corr_df["corr"] > 0.40) & (corr_df["userId"] != random_user)].sort_values("corr", ascending=False)
top_ratings = rating[rating["userId"].isin(top_users["userId"])]
rec = top_ratings.groupby("movieId").agg(mean_rating=("rating", "mean"), count=("rating", "count")).reset_index()
rec = rec[(rec["count"] >= 2) & (rec["mean_rating"] > 3.5)].sort_values("mean_rating", ascending=False)
rec = rec.merge(movie[["movieId", "title"]], on="movieId")
rec = rec[~rec["title"].isin(movies_watched)]
print("\n========== USER BASED (5 Film) ==========")
print(rec[["title", "mean_rating"]].head(5))

# --- ITEM BASED ---
user_ratings = rating[rating["userId"] == random_user].merge(movie[["movieId", "title"]], on="movieId")
user_ratings = user_ratings[user_ratings["title"].isin(user_movie_df.columns)]
last_title = user_ratings[user_ratings["rating"] == 5.0].sort_values("timestamp", ascending=False)["title"].iloc[0]
print(f"\nBaz film: {last_title}")
movie_corr = pd.DataFrame(user_movie_df.corrwith(user_movie_df[last_title]), columns=["corr"]).reset_index()
movie_corr.columns = ["title", "corr"]
movie_corr = movie_corr.sort_values("corr", ascending=False)
print("\n========== ITEM BASED (5 Film) ==========")
print(movie_corr[movie_corr["title"] != last_title].head(5))

Seçilen kullanıcı: 414

========== USER BASED (5 Film) ==========
                                 title  mean_rating
2     Celebration, The (Festen) (1998)     4.750000
3  Guess Who's Coming to Dinner (1967)     4.750000
4                       Yojimbo (1961)     4.750000
5                  12 Angry Men (1957)     4.750000
8                     In Bruges (2008)     4.666667

Baz film: Seven Samurai (Shichinin no samurai) (1954)

========== ITEM BASED (5 Film) ==========
                                title      corr
319                Nine Months (1995)  0.943880
336                 Piano, The (1993)  0.889222
206           Grumpier Old Men (1995)  0.884228
258         King's Speech, The (2010)  0.812421
247  James and the Giant Peach (1996)  0.802955
